# 03 Preprocessing for Click Prediction

## 3.0 Purpose
This notebook prepares the frozen modelling dataset for click prediction. It starts from the saved click-specific recipient-campaign modelling dataframe and defines the evaluation strategy, feature sets, cross-validation structure, and preprocessing rules. No model training is performed in this notebook.

## 3.1 Imports and file paths

In [109]:
# imports
import pandas as pd
import numpy as np

from pathlib import Path

In [110]:
# project paths
PROJECT_ROOT = Path.home() / "Documents" / "Thesis"

DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DATA_DIR = DATA_DIR / "Processed Data"
OUTPUT_DIR = PROJECT_ROOT / "Outputs"

CLICK_MODEL_PATH = PROCESSED_DATA_DIR / "df_click_model_v1.parquet"

print("Project root:", PROJECT_ROOT)
print("Processed data folder exists:", PROCESSED_DATA_DIR.exists())
print("Click model file exists:", CLICK_MODEL_PATH.exists())

Project root: /Users/khanhhien/Documents/Thesis
Processed data folder exists: True
Click model file exists: True


## 3.2 Load frozen modelling dataset

In [111]:
df_click = pd.read_parquet(CLICK_MODEL_PATH)

print("Shape:", df_click.shape)

display(df_click.head())

print("Click distribution:")
display(
    df_click["click"]
    .value_counts(normalize=True)
)

Shape: (1104242, 38)


,user_id,mailing_id,open,click,mailing_info,subject_line,preheader,send_date,send_day_of_week,date_trusted,...,preheader_contains_exclamation,prior_email_count,prior_open_count,prior_click_count,historical_open_rate,historical_click_rate,historical_open_bucket,prior_email_frequency_bucket,had_prior_click,is_first_email
0,5c6bebde5798a794283224c9,668,0.0,0.0,2025/04/24 CPX - MM MAX Magazine - 4 gratis nrs,"Lezen, puzzelen, genieten: 4 edities cadeau","Vraag nu aan – geen kosten, geen verplichtingen.",2025-04-25,Vrijdag,True,...,False,0,0.0,0.0,NaN,NaN,NaN,very_little,False,True
1,5c6bebde5798a794283224c9,691,0.0,0.0,2025/05/29 WOONCOMFORT AIRCO,Bespaar tot 80% op je energiekosten met een airco,Vraag nu gratis maatwerkadvies aan voor energi...,2025-04-30,Woensdag,True,...,False,1,0.0,0.0,0.0,0.0,very_low,very_little,False,False
2,5c6bebde5798a794283224c9,714,0.0,0.0,2025/05/06 CPX - MM - ENGIE BONUS,Profiteer nu: €300 bonus én vaste energietarieven,ENGIE helpt je graag met persoonlijk advies.,2025-05-07,Woensdag,True,...,False,2,0.0,0.0,0.0,0.0,very_low,very_little,False,False
3,5c6bebde5798a794283224c9,733,0.0,0.0,2025/05/14 LIFEPOINTS,Jouw mening is geld waard – start vandaag nog,Verdien punten voor digitale cadeaubonnen zoal...,2025-05-14,Woensdag,True,...,False,3,0.0,0.0,0.0,0.0,very_low,very_little,False,False
4,5c6bebde5798a794283224c9,761,0.0,0.0,2025/05/27 CPX - MM - ENGIE,Speel mee met de ENGIE woordlegger!,"En win een solar powerbank t.w.v. €79,95.",2025-06-04,Woensdag,True,...,False,4,0.0,0.0,0.0,0.0,very_low,very_little,False,False


Click distribution:


click
0.0    0.998918
1.0    0.001082
Name: proportion, dtype: float64

## 3.3 Select modelling target and dataset

In [112]:
# modelling target
TARGET_COL = "click"
GROUP_COL = "mailing_id"

print("Target column:", TARGET_COL)
print("Grouping column:", GROUP_COL)

print()
print("Target distribution:")

display(
    df_click[TARGET_COL]
    .value_counts(normalize=True)
    .sort_index()
)

Target column: click
Grouping column: mailing_id

Target distribution:


click
0.0    0.998918
1.0    0.001082
Name: proportion, dtype: float64

In [113]:
MODELLING_OBJECTIVE = (
    "Predict whether a recipient will click an email "
    "using information available before the campaign is sent."
)

print(MODELLING_OBJECTIVE)

Predict whether a recipient will click an email using information available before the campaign is sent.


## 3.4 Define column roles

Columns are classified by their modelling role. For click prediction, `click` is the target and `open` is excluded because it is a current campaign outcome that would not be known before sending the email.

In [114]:
column_audit = pd.DataFrame({
    "column": df_click.columns,
    "dtype": df_click.dtypes.astype(str).values,
    "missing_rate": df_click.isna().mean().values
})

target_cols = [
    "click"
]

other_target_cols = [
    "open"
]

id_cols = [
    "user_id",
    "mailing_id"
]

audit_only_cols = [
    "mailing_info",
    "subject_line",
    "preheader",
    "send_date",
    "send_day_of_week",
    "date_trusted"
]

candidate_feature_cols = [
    "has_campaign_metadata",
    "gender",
    "age_clean",
    "age_group",
    "n_interests",
    "interest_topic_match",
    "campaign_topic",
    "subject_missing",
    "subject_length",
    "subject_length_group",
    "subject_contains_number",
    "subject_contains_promotion",
    "subject_contains_exclamation",
    "preheader_missing",
    "preheader_length",
    "preheader_length_group",
    "preheader_contains_number",
    "preheader_contains_promotion",
    "preheader_contains_exclamation",
    "prior_email_count",
    "prior_open_count",
    "prior_click_count",
    "historical_open_rate",
    "historical_click_rate",
    "historical_open_bucket",
    "prior_email_frequency_bucket",
    "had_prior_click",
    "is_first_email"
]

def assign_column_role(col):
    if col in target_cols:
        return "target"
    if col in other_target_cols:
        return "other_target_exclude"
    if col in id_cols:
        return "id_exclude"
    if col in audit_only_cols:
        return "audit_exclude"
    if col in candidate_feature_cols:
        return "candidate_feature"
    return "unclassified"

column_audit["role"] = column_audit["column"].apply(assign_column_role)

display(column_audit.sort_values(["role", "column"]))

,column,dtype,missing_rate,role
9,date_trusted,bool,0.000000,audit_exclude
4,mailing_info,object,0.024971,audit_exclude
6,preheader,object,0.029575,audit_exclude
7,send_date,datetime64[us],0.325884,audit_exclude
8,send_day_of_week,object,0.325884,audit_exclude
5,subject_line,object,0.024971,audit_exclude
12,age_clean,float64,0.744601,candidate_feature
13,age_group,category,0.744601,candidate_feature
16,campaign_topic,object,0.000000,candidate_feature
11,gender,object,0.000000,candidate_feature


In [115]:
# validate
display(column_audit["role"].value_counts())

unclassified_cols = column_audit.loc[
    column_audit["role"] == "unclassified",
    "column"
].tolist()

print("Unclassified columns:", unclassified_cols)

role
candidate_feature       28
audit_exclude            6
id_exclude               2
other_target_exclude     1
target                   1
Name: count, dtype: int64

Unclassified columns: []


## 3.5 Define feature sets

Three feature sets are defined for click prediction. The core click feature set is similar to the open feature set, but it also includes historical click behaviour because prior clicking is directly relevant to future clicking. Current `open` is not included because it is a post-send outcome and would cause leakage for pre-send click prediction.

In [116]:
# feature set A - core thesis model
feature_set_A_core = [
    # recipient features
    "gender",
    "age_group",
    "n_interests",
    "interest_topic_match",

    # campaign/content features
    "campaign_topic",
    "has_campaign_metadata",
    "subject_length_group",
    "subject_contains_number",
    "subject_contains_promotion",
    "preheader_length_group",
    "preheader_contains_number",
    "preheader_contains_promotion",

    # historical behaviour features
    "historical_open_bucket",
    "prior_email_frequency_bucket",
    "had_prior_click",
    "is_first_email"
]

In [117]:
# feature set B - core thesis model + interactions
interaction_source_cols = [
    ("campaign_topic", "age_group"),
    ("historical_open_bucket", "prior_email_frequency_bucket")
]

feature_set_B_interactions = feature_set_A_core.copy()

In [118]:
# feature set C - expanded feature set
feature_set_C_expanded = [
    # recipient features
    "gender",
    "age_group",
    "age_clean",
    "n_interests",
    "interest_topic_match",

    # campaign/content features
    "campaign_topic",
    "has_campaign_metadata",
    "subject_missing",
    "subject_length",
    "subject_length_group",
    "subject_contains_number",
    "subject_contains_promotion",
    "subject_contains_exclamation",
    "preheader_missing",
    "preheader_length",
    "preheader_length_group",
    "preheader_contains_number",
    "preheader_contains_promotion",
    "preheader_contains_exclamation",

    # historical behaviour features
    "prior_email_count",
    "prior_open_count",
    "prior_click_count",
    "historical_open_rate",
    "historical_click_rate",
    "historical_open_bucket",
    "prior_email_frequency_bucket",
    "had_prior_click",
    "is_first_email"
]

In [119]:
# validate
feature_sets = {
    "A_core": feature_set_A_core,
    "B_interactions": feature_set_B_interactions,
    "C_expanded": feature_set_C_expanded
}

for name, features in feature_sets.items():
    missing_features = [
        col for col in features
        if col not in df_click.columns
    ]

    print(name)
    print("Number of features:", len(features))
    print("Missing features:", missing_features)
    print()

A_core
Number of features: 16
Missing features: []

B_interactions
Number of features: 16
Missing features: []

C_expanded
Number of features: 28
Missing features: []



## 3.6 Define evaluation and validation strategy

The click prediction task uses the same two-level evaluation structure as open prediction. The latest 20% of campaigns, ordered by `mailing_id`, are reserved as the final untouched test set. The earliest 80% form the train-validation pool.

Because click is extremely rare, inner validation will use StratifiedGroupKFold with `mailing_id` as the grouping variable and `click` as the stratification target. Fold quality must be checked carefully, especially the number of clicked rows in each validation fold.

In [120]:
# evaluation strategy
TARGET_COL = "click"
GROUP_COL = "mailing_id"

FINAL_TEST_SIZE = 0.20
TRAINVAL_SIZE = 1 - FINAL_TEST_SIZE

INNER_CV_SPLITS = 5

print("Target column:", TARGET_COL)
print("Group column:", GROUP_COL)
print("Train-validation share:", TRAINVAL_SIZE)
print("Final test share:", FINAL_TEST_SIZE)
print("Inner CV folds:", INNER_CV_SPLITS)

Target column: click
Group column: mailing_id
Train-validation share: 0.8
Final test share: 0.2
Inner CV folds: 5


## 3.7 Create final chronological campaign holdout

The final holdout split is created at the campaign level. Campaigns are ordered by `mailing_id`, which is used as the available chronological proxy. The earliest 80% of campaigns form the train-validation pool, while the latest 20% are reserved as the final untouched test set.

In [121]:
# campaign-level chronological final holdout
campaign_order = sorted(df_click[GROUP_COL].unique())

n_campaigns = len(campaign_order)
test_start_idx = int(n_campaigns * TRAINVAL_SIZE)

trainval_campaigns = campaign_order[:test_start_idx]
test_campaigns = campaign_order[test_start_idx:]

trainval_df = df_click[df_click[GROUP_COL].isin(trainval_campaigns)].copy()
test_df = df_click[df_click[GROUP_COL].isin(test_campaigns)].copy()

print("Total campaigns:", n_campaigns)
print("Train-validation campaigns:", len(trainval_campaigns))
print("Final test campaigns:", len(test_campaigns))

print("\nTrain-validation shape:", trainval_df.shape)
print("Final test shape:", test_df.shape)

print("\nFirst trainval campaign:", min(trainval_campaigns))
print("Last trainval campaign:", max(trainval_campaigns))
print("First test campaign:", min(test_campaigns))
print("Last test campaign:", max(test_campaigns))

Total campaigns: 274
Train-validation campaigns: 219
Final test campaigns: 55

Train-validation shape: (814269, 38)
Final test shape: (289973, 38)

First trainval campaign: 653
Last trainval campaign: 1252
First test campaign: 1257
Last test campaign: 1418


In [122]:
# safety check
campaign_overlap = set(trainval_campaigns).intersection(set(test_campaigns))

print("Campaign overlap:", len(campaign_overlap))

assert len(campaign_overlap) == 0, "Campaign leakage: some mailing_ids appear in both trainval and test."
assert len(trainval_df) + len(test_df) == len(df_click), "Row counts do not add up."

Campaign overlap: 0


## 3.8 Check final holdout quality

This section checks whether the chronological final holdout split is usable for click prediction. Because clicks are extremely rare, the main concern is not only the click rate but also the absolute number of clicked rows in the train-validation and final test sets.

In [123]:
# basic split summary
split_summary = pd.DataFrame({
    "split": ["trainval", "test"],
    "n_rows": [len(trainval_df), len(test_df)],
    "n_campaigns": [
        trainval_df[GROUP_COL].nunique(),
        test_df[GROUP_COL].nunique()
    ],
    "click_rate": [
        trainval_df[TARGET_COL].mean(),
        test_df[TARGET_COL].mean()
    ],
    "click_positive_count": [
        int(trainval_df[TARGET_COL].sum()),
        int(test_df[TARGET_COL].sum())
    ],
    "click_negative_count": [
        int((trainval_df[TARGET_COL] == 0).sum()),
        int((test_df[TARGET_COL] == 0).sum())
    ],
    "metadata_missing_rate": [
        1 - trainval_df["has_campaign_metadata"].mean(),
        1 - test_df["has_campaign_metadata"].mean()
    ]
})

display(split_summary)

,split,n_rows,n_campaigns,click_rate,click_positive_count,click_negative_count,metadata_missing_rate
0,trainval,814269,219,0.001228,1000,813269,0.033864
1,test,289973,55,0.000672,195,289778,0.000000


In [124]:
# topic distribution
topic_dist = pd.DataFrame({
    "trainval": trainval_df["campaign_topic"].value_counts(normalize=True),
    "test": test_df["campaign_topic"].value_counts(normalize=True)
}).fillna(0)

topic_dist["difference_test_minus_trainval"] = (
    topic_dist["test"] - topic_dist["trainval"]
)

display(topic_dist.sort_values("difference_test_minus_trainval", ascending=False))

,trainval,test,difference_test_minus_trainval
campaign_topic,,,
Media & Publishing,0.369514,0.549675,0.180161
Retail & Promotion,0.001611,0.080387,0.078776
Lottery & Games,0.105620,0.128071,0.022451
Food & Beverages,0.011265,0.017688,0.006423
Finance & Investment,0.012406,0.014732,0.002326
Telecom & Technology,0.001815,0.000000,-0.001815
Business,0.021558,0.006901,-0.014657
Automotive & Mobility,0.228466,0.202546,-0.025920
Health & Wellbeing,0.030866,0.000000,-0.030866


In [125]:
# missingness comparison
missing_compare = pd.DataFrame({
    "trainval_missing": trainval_df.isna().mean(),
    "test_missing": test_df.isna().mean()
})

missing_compare["difference_test_minus_trainval"] = (
    missing_compare["test_missing"] - missing_compare["trainval_missing"]
)

display(
    missing_compare
    .sort_values("difference_test_minus_trainval", ascending=False)
    .head(15)
)

,trainval_missing,test_missing,difference_test_minus_trainval
send_date,0.320511,0.340973,0.020462
send_day_of_week,0.320511,0.340973,0.020462
age_clean,0.741666,0.752843,0.011176
age_group,0.741666,0.752843,0.011176
open,0.082923,0.088019,0.005095
prior_open_count,0.082923,0.088019,0.005095
user_id,0.000000,0.000000,0.000000
subject_contains_exclamation,0.000000,0.000000,0.000000
preheader_missing,0.000000,0.000000,0.000000
preheader_length,0.000000,0.000000,0.000000


The chronological final holdout is retained for click prediction. The split contains 219 train-validation campaigns and 55 final test campaigns, with no campaign overlap. Although the test click rate is lower than the train-validation click rate (0.067% vs. 0.123%), the final test set still contains 195 positive click cases, which is sufficient for later evaluation. The lower test click rate and shifted campaign-topic composition suggest temporal/topic distribution shift, similar to the open prediction split.

The test set contains more Media & Publishing and Retail & Promotion campaigns, while several topics present in train-validation are absent from the test set. Missingness patterns are broadly stable across train-validation and test, with no major new missingness problem. Therefore, the holdout split is considered usable, but final click-model performance should be interpreted under campaign-period distribution shift.

## 3.9 Define inner CV strategy

The train-validation pool uses StratifiedGroupKFold with five folds. Campaigns are treated as groups through `mailing_id`, preventing campaign leakage between training and validation folds. The target variable `click` is used for stratification because clicks are extremely rare and class balance must be maintained across folds as much as possible.

Fold quality will be checked explicitly in the next section before the strategy is accepted.

In [126]:
from sklearn.model_selection import StratifiedGroupKFold

cv_strategy = StratifiedGroupKFold(
    n_splits=INNER_CV_SPLITS,
    shuffle=True,
    random_state=42
)

print(cv_strategy)

X_reference = trainval_df[feature_set_A_core]
y_reference = trainval_df[TARGET_COL]
groups_reference = trainval_df[GROUP_COL]

print("Reference X shape:", X_reference.shape)
print("Reference y shape:", y_reference.shape)
print("Reference groups:", groups_reference.nunique())

StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)
Reference X shape: (814269, 16)
Reference y shape: (814269,)
Reference groups: 219


## 3.10 Check inner CV fold quality

This section checks whether the StratifiedGroupKFold strategy produces usable folds for click prediction. Because clicked rows are extremely rare, the key check is the absolute number of positive clicked rows in each validation fold.

In [127]:
fold_summaries = []

for fold_id, (train_idx, val_idx) in enumerate(
    cv_strategy.split(X_reference, y_reference, groups=groups_reference),
    start=1
):
    fold_train = trainval_df.iloc[train_idx]
    fold_val = trainval_df.iloc[val_idx]

    train_campaigns_fold = set(fold_train[GROUP_COL].unique())
    val_campaigns_fold = set(fold_val[GROUP_COL].unique())
    campaign_overlap = train_campaigns_fold.intersection(val_campaigns_fold)

    fold_summaries.append({
        "fold": fold_id,
        "train_rows": len(fold_train),
        "val_rows": len(fold_val),
        "train_campaigns": fold_train[GROUP_COL].nunique(),
        "val_campaigns": fold_val[GROUP_COL].nunique(),
        "campaign_overlap": len(campaign_overlap),

        "train_click_rate": fold_train[TARGET_COL].mean(),
        "val_click_rate": fold_val[TARGET_COL].mean(),

        "train_click_positive_count": int(fold_train[TARGET_COL].sum()),
        "val_click_positive_count": int(fold_val[TARGET_COL].sum()),

        "val_click_negative_count": int((fold_val[TARGET_COL] == 0).sum())
    })

fold_quality = pd.DataFrame(fold_summaries)

display(fold_quality)

,fold,train_rows,val_rows,train_campaigns,val_campaigns,campaign_overlap,train_click_rate,val_click_rate,train_click_positive_count,val_click_positive_count,val_click_negative_count
0,1,650238,164031,176,43,0,0.001143,0.001567,743,257,163774
1,2,650469,163800,175,44,0,0.001314,0.000885,855,145,163655
2,3,633122,181147,175,44,0,0.001395,0.000646,883,117,181030
3,4,666325,147944,176,43,0,0.001108,0.001771,738,262,147682
4,5,656922,157347,174,45,0,0.001189,0.001392,781,219,157128


In [128]:
assert (fold_quality["campaign_overlap"] == 0).all(), "Campaign leakage found in CV folds."
assert (fold_quality["val_click_positive_count"] > 0).all(), "Some validation folds have no clicked cases."
assert (fold_quality["val_click_negative_count"] > 0).all(), "Some validation folds have no non-click cases."

The StratifiedGroupKFold strategy was retained. No campaign leakage was observed across folds. All validation folds contained both positive and negative click observations, and each validation fold contained at least 117 positive click cases. Although click rates vary across folds, this is expected given the extreme rarity of click events and campaign-level grouping constraints. The fold structure is therefore considered suitable for model development and hyperparameter tuning.

## 3.11 Create optional interaction columns

Two deterministic interaction columns are created for the interaction feature set. These interactions mirror the open prediction setup and are theoretically motivated by campaign-topic differences across age groups and by the relationship between historical engagement level and prior email frequency.

No interaction involving the current `open` outcome is created, because current open status is not available before sending and would cause leakage for pre-send click prediction.

In [129]:
def combine_interaction_cols(df, col1, col2):
    return (
        df[col1].astype("object").fillna("Missing").astype(str)
        + " x "
        + df[col2].astype("object").fillna("Missing").astype(str)
    )

for data in [trainval_df, test_df]:
    data["campaign_topic_x_age_group"] = combine_interaction_cols(
        data,
        "campaign_topic",
        "age_group"
    )

    data["historical_open_bucket_x_prior_email_frequency_bucket"] = combine_interaction_cols(
        data,
        "historical_open_bucket",
        "prior_email_frequency_bucket"
    )

In [130]:
# validate
interaction_cols = [
    "campaign_topic_x_age_group",
    "historical_open_bucket_x_prior_email_frequency_bucket"
]

for col in interaction_cols:
    print(col)
    print("Trainval unique values:", trainval_df[col].nunique())
    print("Test unique values:", test_df[col].nunique())
    print("Trainval missing:", trainval_df[col].isna().mean())
    print("Test missing:", test_df[col].isna().mean())
    print()

campaign_topic_x_age_group
Trainval unique values: 60
Test unique values: 35
Trainval missing: 0.0
Test missing: 0.0

historical_open_bucket_x_prior_email_frequency_bucket
Trainval unique values: 30
Test unique values: 30
Trainval missing: 0.0
Test missing: 0.0



## 3.12 Update feature sets with interaction columns

In [131]:
feature_set_A_core = [
    "age_group",
    "gender",

    "campaign_topic",

    "n_interests",
    "interest_topic_match",

    "prior_email_frequency_bucket",
    "historical_open_bucket",

    "had_prior_click",

    "has_campaign_metadata",

    "subject_length_group",
    "preheader_length_group",

    "subject_contains_number",
    "subject_contains_promotion",

    "preheader_contains_number",
    "preheader_contains_promotion",

    "is_first_email"
]

In [132]:
feature_set_B_interactions = (
    feature_set_A_core
    + [
        "campaign_topic_x_age_group",
        "historical_open_bucket_x_prior_email_frequency_bucket"
    ]
)

In [133]:
feature_set_C_expanded = [
    "age_clean",
    "age_group",
    "gender",

    "campaign_topic",

    "n_interests",
    "interest_topic_match",

    "subject_length",
    "subject_length_group",
    "subject_missing",

    "preheader_length",
    "preheader_length_group",
    "preheader_missing",

    "subject_contains_number",
    "subject_contains_promotion",
    "subject_contains_exclamation",

    "preheader_contains_number",
    "preheader_contains_promotion",
    "preheader_contains_exclamation",

    "prior_email_count",
    "prior_email_frequency_bucket",

    "prior_open_count",
    "prior_click_count",

    "historical_open_rate",
    "historical_open_bucket",

    "historical_click_rate",
    "had_prior_click",

    "has_campaign_metadata",

    "is_first_email"
]

In [134]:
# validation

feature_sets = {
    "A_core": feature_set_A_core,
    "B_interactions": feature_set_B_interactions,
    "C_expanded": feature_set_C_expanded
}

for name, features in feature_sets.items():

    missing_features = [
        col for col in features
        if col not in trainval_df.columns
    ]

    print(name)
    print("Number of features:", len(features))
    print("Missing features:", missing_features)
    print()

A_core
Number of features: 16
Missing features: []

B_interactions
Number of features: 18
Missing features: []

C_expanded
Number of features: 28
Missing features: []



## 3.13 Define feature types by feature set

Features are classified into numerical, categorical, and binary groups so that different preprocessing rules can be applied later.

In [135]:
feature_types_A = {
    "numeric": [
        "n_interests"
    ],

    "categorical": [
        "age_group",
        "gender",
        "campaign_topic",
        "prior_email_frequency_bucket",
        "historical_open_bucket",
        "subject_length_group",
        "preheader_length_group"
    ],

    "binary": [
        "interest_topic_match",
        "had_prior_click",
        "has_campaign_metadata",
        "subject_contains_number",
        "subject_contains_promotion",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "is_first_email"
    ]
}

In [136]:
feature_types_B = {
    "numeric": feature_types_A["numeric"].copy(),

    "categorical": (
        feature_types_A["categorical"].copy()
        + [
            "campaign_topic_x_age_group",
            "historical_open_bucket_x_prior_email_frequency_bucket"
        ]
    ),

    "binary": feature_types_A["binary"].copy()
}

In [137]:
feature_types_C = {
    "numeric": [
        "age_clean",
        "n_interests",
        "subject_length",
        "preheader_length",
        "prior_email_count",
        "prior_open_count",
        "prior_click_count",
        "historical_open_rate",
        "historical_click_rate"
    ],

    "categorical": [
        "age_group",
        "gender",
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "prior_email_frequency_bucket",
        "historical_open_bucket"
    ],

    "binary": [
        "interest_topic_match",
        "subject_missing",
        "preheader_missing",
        "subject_contains_number",
        "subject_contains_promotion",
        "subject_contains_exclamation",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "preheader_contains_exclamation",
        "had_prior_click",
        "has_campaign_metadata",
        "is_first_email"
    ]
}

In [138]:
# validate
feature_type_sets = {
    "A_core": feature_types_A,
    "B_interactions": feature_types_B,
    "C_expanded": feature_types_C
}

for name, type_dict in feature_type_sets.items():
    typed_features = (
        type_dict["numeric"]
        + type_dict["categorical"]
        + type_dict["binary"]
    )

    original_features = feature_sets[name]

    missing_from_types = sorted(set(original_features) - set(typed_features))
    extra_in_types = sorted(set(typed_features) - set(original_features))

    duplicate_features = pd.Series(typed_features).value_counts()
    duplicate_features = duplicate_features[duplicate_features > 1].index.tolist()

    print(name)
    print("Original feature count:", len(original_features))
    print("Typed feature count:", len(typed_features))
    print("Missing from type definitions:", missing_from_types)
    print("Extra in type definitions:", extra_in_types)
    print("Duplicated in type definitions:", duplicate_features)
    print()

A_core
Original feature count: 16
Typed feature count: 16
Missing from type definitions: []
Extra in type definitions: []
Duplicated in type definitions: []

B_interactions
Original feature count: 18
Typed feature count: 18
Missing from type definitions: []
Extra in type definitions: []
Duplicated in type definitions: []

C_expanded
Original feature count: 28
Typed feature count: 28
Missing from type definitions: []
Extra in type definitions: []
Duplicated in type definitions: []



## 3.14 Define preprocessing rules

This section defines the preprocessing rules for click prediction. The rules are defined here but are not fitted globally. During modelling, preprocessing must be fitted only on the training fold inside cross-validation, then applied to validation or test data.

In [139]:
preprocessing_rules = {
    "numeric": {
        "missing_values": "impute with 0",
        "scaling": "standardize using training-fold mean and standard deviation",
        "reason": (
            "Numerical variables are scaled for logistic regression. "
            "Missing historical values are imputed with 0 where this represents no prior history, "
            "while first-email status remains available through is_first_email."
        )
    },

    "categorical": {
        "missing_values": "impute with 'Missing'",
        "encoding": "one-hot encode",
        "unknown_categories": "ignore unseen categories in validation/test",
        "reason": (
            "Categorical missingness may be informative. "
            "One-hot encoding is required for logistic regression, and unseen categories must not break validation/test transformation."
        )
    },

    "binary": {
        "missing_values": "impute with most frequent value if needed",
        "encoding": "pass through as 0/1",
        "reason": (
            "Binary indicators are already model-ready and do not need scaling."
        )
    }
}

preprocessing_rules

{'numeric': {'missing_values': 'impute with 0',
  'scaling': 'standardize using training-fold mean and standard deviation',
  'reason': 'Numerical variables are scaled for logistic regression. Missing historical values are imputed with 0 where this represents no prior history, while first-email status remains available through is_first_email.'},
 'categorical': {'missing_values': "impute with 'Missing'",
  'encoding': 'one-hot encode',
  'unknown_categories': 'ignore unseen categories in validation/test',
  'reason': 'Categorical missingness may be informative. One-hot encoding is required for logistic regression, and unseen categories must not break validation/test transformation.'},
 'binary': {'missing_values': 'impute with most frequent value if needed',
  'encoding': 'pass through as 0/1',
  'reason': 'Binary indicators are already model-ready and do not need scaling.'}}

In [140]:
for name, type_dict in feature_type_sets.items():
    print(name)

    for feature_type, cols in type_dict.items():
        missing_summary = (
            trainval_df[cols]
            .isna()
            .mean()
            .sort_values(ascending=False)
        )

        print(f"\n{feature_type} missingness:")
        display(missing_summary)

    print("\n" + "-" * 60 + "\n")

A_core

numeric missingness:


n_interests    0.0
dtype: float64


categorical missingness:


age_group                       0.741666
historical_open_bucket          0.103138
gender                          0.000000
campaign_topic                  0.000000
prior_email_frequency_bucket    0.000000
subject_length_group            0.000000
preheader_length_group          0.000000
dtype: float64


binary missingness:


...


categorical missingness:


age_group                       0.741666
historical_open_bucket          0.103138
gender                          0.000000
campaign_topic                  0.000000
subject_length_group            0.000000
preheader_length_group          0.000000
prior_email_frequency_bucket    0.000000
dtype: float64


binary missingness:


interest_topic_match              0.0
subject_missing                   0.0
preheader_missing                 0.0
subject_contains_number           0.0
subject_contains_promotion        0.0
subject_contains_exclamation      0.0
preheader_contains_number         0.0
preheader_contains_promotion      0.0
preheader_contains_exclamation    0.0
had_prior_click                   0.0
has_campaign_metadata             0.0
is_first_email                    0.0
dtype: float64


------------------------------------------------------------



Missingness patterns are consistent with feature construction logic. Missing historical engagement variables correspond primarily to users without sufficient prior behavioural history, while demographic missingness originates from unavailable age information. These values will be handled through preprocessing pipelines rather than manual imputation.

## 3.15 Build preprocessing pipeline definitions

This section defines reusable preprocessing pipelines for each click prediction feature set. These preprocessing objects are not fitted globally in this notebook. During modelling, they must be fitted only on the training fold inside cross-validation.

In [141]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [142]:
def build_preprocessor(feature_types):
    numeric_features = feature_types["numeric"]
    categorical_features = feature_types["categorical"]
    binary_features = feature_types["binary"]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    binary_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
            ("binary", binary_transformer, binary_features)
        ],
        remainder="drop"
    )

    return preprocessor

In [143]:
preprocessors = {
    "A_core": build_preprocessor(feature_types_A),
    "B_interactions": build_preprocessor(feature_types_B),
    "C_expanded": build_preprocessor(feature_types_C)
}

for name, preprocessor in preprocessors.items():
    print(name)
    print(preprocessor)
    print()

A_core
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=0,
                                                                strategy='constant')),
                                                 ('scaler', StandardScaler())]),
                                 ['n_interests']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['age_group', 'gender', 'campaign_t...
                                  'historical_open_bucket',
                        

## 3.16 Save preprocessing metadata

This section saves the click preprocessing metadata for reproducibility. No preprocessing is fitted or applied here. The saved metadata records the target, grouping variable, holdout strategy, cross-validation strategy, feature sets, feature types, and preprocessing rules.

In [144]:
import json

click_preprocessing_metadata = {
    "target": TARGET_COL,
    "group_column": GROUP_COL,
    "final_test_size": FINAL_TEST_SIZE,
    "trainval_size": TRAINVAL_SIZE,
    "inner_cv_splits": INNER_CV_SPLITS,
    "cv_strategy": "StratifiedGroupKFold",
    "cv_shuffle": True,
    "cv_random_state": 42,

    "feature_sets": feature_sets,
    "feature_types": feature_type_sets,

    "preprocessing_rules": preprocessing_rules,

    "notes": {
        "objective": MODELLING_OBJECTIVE,
        "holdout": (
            "The latest 20% of campaigns by mailing_id are reserved as the final untouched test set."
        ),
        "grouping": (
            "mailing_id is used as the grouping variable to prevent campaign leakage."
        ),
        "click_imbalance": (
            "Click prediction is an extreme rare-event task. Fold quality must be checked using absolute positive click counts."
        ),
        "open_exclusion": (
            "Current open is excluded from click prediction because it is a post-send outcome and would cause leakage for pre-send prediction."
        )
    }
}

In [145]:
# validate
print("Target:", click_preprocessing_metadata["target"])
print("Group:", click_preprocessing_metadata["group_column"])
print("CV:", click_preprocessing_metadata["cv_strategy"])
print()

for name, features in click_preprocessing_metadata["feature_sets"].items():
    print(name)
    print("Features:", len(features))
    print()

Target: click
Group: mailing_id
CV: StratifiedGroupKFold

A_core
Features: 16

B_interactions
Features: 18

C_expanded
Features: 28



In [146]:
CLICK_CONFIG_PATH = OUTPUT_DIR / "click_preprocessing_metadata.json"

with open(CLICK_CONFIG_PATH, "w") as f:
    json.dump(click_preprocessing_metadata, f, indent=4)

print("Saved to:", CLICK_CONFIG_PATH)
print("File exists:", CLICK_CONFIG_PATH.exists())

Saved to: /Users/khanhhien/Documents/Thesis/Outputs/click_preprocessing_metadata.json
File exists: True
